In [ ]:
def doolittle_lu(A):
    n = len(A)
    
    # Initialize L and U as zero matrices
    L = [[0.0] * n for _ in range(n)]
    U = [[0.0] * n for _ in range(n)]
    
    # Doolittle algorithm
    for i in range(n):
        # 1️⃣ Compute U (upper triangular)
        for j in range(i, n):
            sum_u = sum(L[i][k] * U[k][j] for k in range(i))
            U[i][j] = A[i][j] - sum_u
        
        # 2️⃣ Set diagonal of L = 1
        L[i][i] = 1.0
        
        # 3️⃣ Compute L (lower triangular)
        for j in range(i+1, n):
            sum_l = sum(L[j][k] * U[k][i] for k in range(i))
            L[j][i] = (A[j][i] - sum_l) / U[i][i]
    
    return L, U


In [ ]:
A = [
    [2, 3, 1],
    [4, 7, 3],
    [6, 18, 5]
]

L, U = doolittle_lu(A)

print("L =")
for row in L:
    print(row)

print("\nU =")
for row in U:
    print(row)


In [ ]:
def matmul(X, Y):
    n = len(X)
    Z = [[0.0]*n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            Z[i][j] = sum(X[i][k] * Y[k][j] for k in range(n))
    return Z

LU = matmul(L, U)

print("\nLU =")
for row in LU:
    print(row)


In [ ]:
def crout_lu(A):
    n = len(A)
    L = [[0.0]*n for _ in range(n)]
    U = [[0.0]*n for _ in range(n)]
    
    for j in range(n):
        U[j][j] = 1.0  # Unit diagonal of U
        
        for i in range(j, n):
            L[i][j] = A[i][j] - sum(L[i][k]*U[k][j] for k in range(j))
        
        for i in range(j+1, n):
            U[j][i] = (A[j][i] - sum(L[j][k]*U[k][i] for k in range(j))) / L[j][j]
    
    return L, U


In [ ]:
import numpy as np

def cholesky_decomposition(A):
    n = len(A)
    L = np.zeros_like(A, dtype=float)
    
    for i in range(n):
        for j in range(i + 1):
            sum_val = sum(L[i][k] * L[j][k] for k in range(j))
            
            if i == j:
                L[i][j] = np.sqrt(A[i][i] - sum_val)
            else:
                L[i][j] = (A[i][j] - sum_val) / L[j][j]
    
    return L

# Example matrix
A = np.array([
    [4, 12, -16],
    [12, 37, -43],
    [-16, -43, 98]
], dtype=float)

L = cholesky_decomposition(A)
print("L =\n", L)
print("Check LL^T =\n", L @ L.T)


In [ ]:
import numpy as np

def cholesky_decomposition(A):
    """
    Manual Cholesky decomposition: return lower-triangular L such that A = L @ L.T
    A must be symmetric positive definite.
    """
    A = np.array(A, dtype=float)
    n = A.shape[0]
    L = np.zeros_like(A)
    
    for i in range(n):
        for j in range(i+1):
            sum_val = sum(L[i, k] * L[j, k] for k in range(j))
            if i == j:
                val = A[i, i] - sum_val
                if val <= 0:
                    raise np.linalg.LinAlgError(
                        f"Matrix is not positive definite at diagonal {i}: {val}"
                    )
                L[i, j] = np.sqrt(val)
            else:
                L[i, j] = (A[i, j] - sum_val) / L[j, j]
    return L

def forward_substitution(L, b):
    """
    Solve Ly = b for y (L lower-triangular, unit or non-unit diagonal allowed).
    """
    n = L.shape[0]
    y = np.zeros(n, dtype=float)
    for i in range(n):
        y[i] = (b[i] - np.dot(L[i, :i], y[:i])) / (L[i, i])
    return y

def backward_substitution(U, y):
    """
    Solve Ux = y for x (U upper-triangular).
    """
    n = U.shape[0]
    x = np.zeros(n, dtype=float)
    for i in range(n-1, -1, -1):
        x[i] = (y[i] - np.dot(U[i, i+1:], x[i+1:])) / U[i, i]
    return x

def solve_cholesky(A, b, use_numpy=False):
    """
    Solve Ax = b using Cholesky:
      - If use_numpy True, uses np.linalg.cholesky for L
      - Otherwise uses manual cholesky_decomposition
    """
    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float)
    if use_numpy:
        L = np.linalg.cholesky(A)
    else:
        L = cholesky_decomposition(A)
    
    # solve L y = b
    y = forward_substitution(L, b)
    # solve L.T x = y
    x = backward_substitution(L.T, y)
    return x, L

# -------------------------
# Example
# -------------------------
A = np.array([[4, 12, -16],
              [12, 37, -43],
              [-16, -43, 98]], dtype=float)
b = np.array([1, 2, 3], dtype=float)

# Using manual cholesky
x_manual, L_manual = solve_cholesky(A, b, use_numpy=False)
print("Manual Cholesky L:\n", L_manual)
print("Solution x (manual):", x_manual)

# Using numpy cholesky
x_np, L_np = solve_cholesky(A, b, use_numpy=True)
print("\nNumPy Cholesky L:\n", L_np)
print("Solution x (numpy):", x_np)

# Verify
print("\nVerify A @ x ≈ b (manual):", np.allclose(A @ x_manual, b))
print("Verify A @ x ≈ b (numpy):", np.allclose(A @ x_np, b))
